# Data Processing

In [ ]:
# Run only if country_converter is not installed:
# %pip install country_converter

Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
import numpy as np
import pandas as pd
import country_converter as coco
from sklearn.preprocessing import StandardScaler

In [ ]:
OUTPUT_DIR = "outputs"
TABLES_DIR = os.path.join(OUTPUT_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")

# INPUT_FILE = os.path.join(OUTPUT_DIR, "preprocessed__middle_east_master.csv")
INPUT_FILE = os.path.join(OUTPUT_DIR, "preprocessed_all_master.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [95]:
# load datatset
df_raw = pd.read_csv(INPUT_FILE)

print("Raw dataset shape:", df_raw.shape)
print("\nOriginal columns:")
print(df_raw.columns.tolist())

Raw dataset shape: (186, 12)

Original columns:
['country_name', 'country_code', 'obesity_value', 'obesity_high', 'obesity_low', 'activity_value', 'activity_high', 'activity_low', 'gdp_per_capita_2022', 'asr_world', 'new_cases', 'crude_rate']


In [96]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
)

print("\nCleaned columns:")
print(df.columns.tolist())


Cleaned columns:
['country_name', 'country_code', 'obesity_value', 'obesity_high', 'obesity_low', 'activity_value', 'activity_high', 'activity_low', 'gdp_per_capita_2022', 'asr_world', 'new_cases', 'crude_rate']


In [97]:
country_col = "country_name"
iso3_col = "country_code"
cancer_col = "asr_world"
obesity_col = "obesity_value"
activity_col = "activity_value"
gdp_col = "gdp_per_capita_2022"

# Convert key variables to numeric


numeric_cols = [
    cancer_col,
    obesity_col,
    activity_col,
    gdp_col,
    "new_cases",
    "crude_rate"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [98]:
# dereived avriables

# Main cancer index used in the analysis
df["cancer_index"] = df[cancer_col]

# Natural log of GDP per capita
df["log_gdp"] = np.log(df[gdp_col])

# Middle East flag
middle_east_iso3 = {
    "BHR", "CYP", "EGY", "IRN", "IRQ", "ISR", "JOR", "KWT",
    "LBN", "OMN", "PSE", "QAT", "SAU", "SYR", "TUR", "ARE", "YEM"
}

df["middle_east_flag"] = df[iso3_col].isin(middle_east_iso3)

In [99]:
# Add region labels

# Convert ISO3 country codes to broad continent labels
df["continent"] = coco.convert(
    names=df[iso3_col].tolist(),
    src="ISO3",
    to="continent"
)

# Convert continent labels into paper-friendly region labels
continent_to_region = {
    "Africa": "Africa",
    "Europe": "Europe",
    "Asia": "Asia",
    "America": "Americas",
    "Oceania": "Oceania"
}

df["region"] = df["continent"].map(continent_to_region)

# Override Middle Eastern countries into a separate Middle East region
df.loc[df["middle_east_flag"], "region"] = "Middle East"

# Check whether any country failed region assignment
missing_region = df[
    df["region"].isna() | (df["continent"] == "not found")
][[country_col, iso3_col, "continent", "region"]]

print("Countries with missing region labels:")
display(missing_region)

print("\nRegion counts:")
display(df["region"].value_counts())

nan not found in ISO3


Countries with missing region labels:


,country_name,country_code,continent,region
185,Total,NaN,not found,NaN



Region counts:


Africa         53
Europe         39
Americas       34
Asia           32
Middle East    17
Oceania        10
Name: region, dtype: int64

### Complete case Filtyering (missing data)

In [100]:
complete_case_cols = [
    country_col,
    iso3_col,
    cancer_col,
    obesity_col,
    activity_col,
    gdp_col,
    "log_gdp",
    "region"
]

n_before = len(df)

missing_before = df[complete_case_cols].isna().sum().reset_index()
missing_before.columns = ["variable", "missing_values_before_filtering"]

df = df.dropna(subset=complete_case_cols).copy()

n_after = len(df)

print("\nComplete-case filtering:")
print("Countries before filtering:", n_before)
print("Countries after filtering:", n_after)
print("Countries dropped:", n_before - n_after)


Complete-case filtering:
Countries before filtering: 186
Countries after filtering: 173
Countries dropped: 13


In [101]:
# Varaibles for PCA, clustering, and heatmaps

pca_vars = [
    "cancer_index",
    obesity_col,
    activity_col,
    "log_gdp"
]

z_cols = [
    "z_cancer_index",
    "z_obesity",
    "z_physical_activity",
    "z_log_gdp"
]

scaler = StandardScaler()
df[z_cols] = scaler.fit_transform(df[pca_vars])

In [102]:
# output directories

os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)


# EDA

In [103]:
# Dataset Summary


# Count regions
n_regions = df["region"].nunique()

# Count Middle East countries
n_middle_east = df["middle_east_flag"].sum()

# Main dataset summary rows
dataset_summary_rows = [
    {
        "section": "Dataset size",
        "item": "Countries before complete-case filtering",
        "value": n_before
    },
    {
        "section": "Dataset size",
        "item": "Countries after complete-case filtering",
        "value": n_after
    },
    {
        "section": "Dataset size",
        "item": "Countries dropped due to missing core variables",
        "value": n_before - n_after
    },
    {
        "section": "Regional structure",
        "item": "Number of regions",
        "value": n_regions
    },
    {
        "section": "Regional structure",
        "item": "Region categories",
        "value": ", ".join(sorted(df["region"].dropna().unique()))
    },
    {
        "section": "Middle East subgroup",
        "item": "Middle East countries in final dataset",
        "value": int(n_middle_east)
    }
]

# Variable/unit rows
variable_unit_rows = [
    {
        "section": "Variable and unit",
        "item": "country_name",
        "value": "Country name"
    },
    {
        "section": "Variable and unit",
        "item": "country_code",
        "value": "ISO3 country code"
    },
    {
        "section": "Variable and unit",
        "item": "asr_world / cancer_index",
        "value": "Colorectal cancer incidence ASR, World standard population, per 100,000"
    },
    {
        "section": "Variable and unit",
        "item": "obesity_value",
        "value": "Adult obesity prevalence, %, age 18+, 2022"
    },
    {
        "section": "Variable and unit",
        "item": "activity_value",
        "value": "Adult insufficient physical activity prevalence, %, age 18+, 2022"
    },
    {
        "section": "Variable and unit",
        "item": "gdp_per_capita_2022",
        "value": "GDP per capita, current US$, 2022"
    },
    {
        "section": "Variable and unit",
        "item": "log_gdp",
        "value": "Natural logarithm of GDP per capita"
    },
    {
        "section": "Variable and unit",
        "item": "region",
        "value": "Broad world region used for regional comparison"
    },
    {
        "section": "Variable and unit",
        "item": "middle_east_flag",
        "value": "Boolean indicator for Middle Eastern countries"
    }
]

# Combine everything into one Table 1
eda_table_1 = pd.DataFrame(
    dataset_summary_rows + variable_unit_rows
)

# Preview
display(eda_table_1)

,section,item,value
0,Dataset size,Countries before complete-case filtering,186
1,Dataset size,Countries after complete-case filtering,173
2,Dataset size,Countries dropped due to missing core variables,13
3,Regional structure,Number of regions,6
4,Regional structure,Region categories,"Africa, Americas, Asia, Europe, Middle East, O..."
5,Middle East subgroup,Middle East countries in final dataset,16
6,Variable and unit,country_name,Country name
7,Variable and unit,country_code,ISO3 country code
8,Variable and unit,asr_world / cancer_index,"Colorectal cancer incidence ASR, World standar..."
9,Variable and unit,obesity_value,"Adult obesity prevalence, %, age 18+, 2022"


# Descriptive statsistcis

In [104]:
desc_vars = [
    cancer_col,
    obesity_col,
    activity_col,
    gdp_col,
    "log_gdp"
]

desc_rows = []

for col in desc_vars:
    s = df[col].dropna()
    desc_rows.append({
        "variable": col,
        "mean": s.mean(),
        "median": s.median(),
        "standard_deviation": s.std(ddof=1),
        "min": s.min(),
        "max": s.max(),
        "iqr": s.quantile(0.75) - s.quantile(0.25)
    })

desc_table = pd.DataFrame(desc_rows)

# Preview only
display(desc_table.round(3))

,variable,mean,median,standard_deviation,min,max,iqr
0,asr_world,49.212,42.600,23.719,9.500,166.700,35.400
1,obesity_value,21.832,21.780,10.891,2.020,62.430,17.080
2,activity_value,72.871,75.720,13.091,33.860,97.300,19.640
3,gdp_per_capita_2022,16674.821,6515.585,23570.366,302.993,123719.659,18485.243
4,log_gdp,8.802,8.782,1.435,5.714,11.726,2.189


In [ ]:

# Save the final processed dataset
# processed_path = os.path.join(OUTPUT_DIR, "processed_middle_east_master.csv")
processed_path = os.path.join(OUTPUT_DIR, "processed_all_master.csv")
df.to_csv(processed_path, index=False)

# Save EDA Table 1: Dataset Summary

eda_table_1_csv_path = os.path.join(TABLES_DIR, "eda_table_1_dataset_summary.csv")
eda_table_1_tex_path = os.path.join(TABLES_DIR, "eda_table_1_dataset_summary.tex")

eda_table_1.to_csv(
    eda_table_1_csv_path,
    index=False
)

eda_table_1.to_latex(
    eda_table_1_tex_path,
    index=False
)

# Save EDA Table 2: Descriptive Statistics

eda_table_2_csv_path = os.path.join(TABLES_DIR, "eda_table_2_descriptive_statistics.csv")
eda_table_2_tex_path = os.path.join(TABLES_DIR, "eda_table_2_descriptive_statistics.tex")

desc_table.to_csv(
    eda_table_2_csv_path,
    index=False
)

desc_table.round(3).to_latex(
    eda_table_2_tex_path,
    index=False
)


print("\nFinal processed dataset saved to:")
print(os.path.abspath(processed_path))
print("File exists?", os.path.exists(processed_path))

print("\nEDA Table 1 saved to:")
print(os.path.abspath(eda_table_1_csv_path))
print(os.path.abspath(eda_table_1_tex_path))

print("\nEDA Table 2 saved to:")
print(os.path.abspath(eda_table_2_csv_path))
print(os.path.abspath(eda_table_2_tex_path))

print("\nFinal shape:", df.shape)

print("\nFinal columns added:")
print([
    "cancer_index",
    "log_gdp",
    "middle_east_flag",
    "continent",
    "region",
    "z_cancer_index",
    "z_obesity",
    "z_physical_activity",
    "z_log_gdp"
])

print("\nFiles inside outputs folder:")
print(os.listdir("outputs"))

print("\nFiles inside outputs/tables folder:")
print(os.listdir("outputs/tables"))

df.head()


Final processed dataset saved to:
d:\SQU\Data Exploration\Project\Current\outputs\processed_global_obesity_cancer_2022.csv
File exists? True

EDA Table 1 saved to:
d:\SQU\Data Exploration\Project\Current\outputs\tables\eda_table_1_dataset_summary.csv
d:\SQU\Data Exploration\Project\Current\outputs\tables\eda_table_1_dataset_summary.tex

EDA Table 2 saved to:
d:\SQU\Data Exploration\Project\Current\outputs\tables\eda_table_2_descriptive_statistics.csv
d:\SQU\Data Exploration\Project\Current\outputs\tables\eda_table_2_descriptive_statistics.tex

Final shape: (173, 21)

Final columns added:
['cancer_index', 'log_gdp', 'middle_east_flag', 'continent', 'region', 'z_cancer_index', 'z_obesity', 'z_physical_activity', 'z_log_gdp']

Files inside outputs folder:
['figures', 'processed_global_obesity_cancer_2022.csv', 'tables']

Files inside outputs/tables folder:
['eda_table_1_dataset_summary.csv', 'eda_table_1_dataset_summary.tex', 'eda_table_2_descriptive_statistics.csv', 'eda_table_2_descrip

C:\Users\Raihan Karim Ishmam\AppData\Local\Temp\ipykernel_21656\1931059987.py:16: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  eda_table_1.to_latex(
C:\Users\Raihan Karim Ishmam\AppData\Local\Temp\ipykernel_21656\1931059987.py:31: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  desc_table.round(3).to_latex(


,country_name,country_code,obesity_value,obesity_high,obesity_low,activity_value,activity_high,activity_low,gdp_per_capita_2022,asr_world,...,crude_rate,cancer_index,log_gdp,middle_east_flag,continent,region,z_cancer_index,z_obesity,z_physical_activity,z_log_gdp
0,Afghanistan,AFG,19.22,22.20,16.33,66.64,76.96,55.78,357.261153,32.7,...,16.1,32.7,5.878467,False,Asia,Asia,-0.698173,-0.240477,-0.477345,-2.043674
1,Albania,ALB,23.36,26.48,20.43,75.72,90.95,54.11,7756.961887,39.5,...,74.6,39.5,8.956346,False,Europe,Europe,-0.410653,0.140750,0.218271,0.107761
2,Algeria,DZA,23.81,27.25,20.51,70.96,80.97,60.12,4960.303343,39.4,...,39.6,39.4,8.509222,False,Africa,Africa,-0.414881,0.182188,-0.146391,-0.204778
3,Angola,AGO,11.47,16.03,7.63,82.67,94.51,63.58,3682.113151,27.4,...,13.4,27.4,8.211242,False,Africa,Africa,-0.922270,-0.954128,0.750709,-0.413066
4,Azerbaijan,AZE,26.55,29.97,23.21,76.08,84.68,65.52,7770.594223,54.5,...,64.6,54.5,8.958102,False,Asia,Asia,0.223584,0.434498,0.245851,0.108988
